# PPO Implementation Architecture

## Overview

Now we translate the concepts into **actual tensors**. Still no code yet.

Suppose we collect $T$ timesteps.

## What We Store in the Rollout Buffer

For every timestep we store:

| Tensor | Meaning |
|---|---|
| `states` | State seen by actor/critic |
| `actions` | Action actor actually selected |
| `rewards` | Environment reward |
| `dones` | Whether episode ended |
| `old_log_probs` | Probability of selected action under old policy |
| `values` | Critic's $V(\text{state})$ |

### Example Row

```
state          = S3
action         = RIGHT
reward         = +2
done           = False
old_log_prob   = -0.51
value          = 4.2
```

---

## 1. After Rollout: Calculate Advantages

We already have:

```
rewards
values
```

We **calculate TD errors**:

$$\delta_t = \text{reward} + \gamma \times V(\text{next\_state}) - V(\text{state})$$

Then **GAE turns those into advantages**:

$$A = [1.8, 0.4, -0.7, 2.1, \ldots]$$

### Interpretation

| Value | Meaning |
|---|---|
| Positive | Chosen action was better than expected |
| Negative | Chosen action was worse than expected |

## 2. Calculate Returns for the Critic

We also need a **training target for the critic**.

**Conceptually:**

$$\text{return} = \text{estimated total reward from this state}$$

So we get:

$$\text{returns} = [7.2, 6.1, 5.4, 3.2, \ldots]$$

**Key Distinction:**

| Network | Target |
|---|---|
| **Critic** | `critic prediction → return` |
| **Actor** | Uses `advantage` |

These are **different targets** for different networks.

## 3. Create Minibatches

Suppose we collected:

$$T = 2048 \text{ transitions}$$

We **don't feed all 2048 into the optimizer at once**.

We **shuffle and split**:

```
2048 transitions
       ↓
256
256
256
...
```

Each minibatch updates both actor and critic.

## 4. Actor Calculation

For a minibatch, **run the current actor** on the stored states.

We get: `new_log_probs` for the actions that were actually taken.

Then:

$$\text{ratio} = e^{\text{new\_log\_prob} - \text{old\_log\_prob}}$$

### Why Log Probabilities?

Because:

$$\frac{P_{\text{new}}}{P_{\text{old}}} = e^{\log P_{\text{new}} - \log P_{\text{old}}}$$

**Numerically, this is much more stable** than computing probabilities directly.

## 5. PPO Actor Loss

We now have:

```
ratio
advantage
```

**Calculate:**

$$\text{unclipped} = \text{ratio} \times \text{advantage}$$

$$\text{clipped} = \text{clip}(\text{ratio}, 1-\epsilon, 1+\epsilon) \times \text{advantage}$$

Then:

$$\text{objective} = \min(\text{unclipped}, \text{clipped})$$

**For optimization:**

$$\text{actor\_loss} = -\text{mean}(\text{objective})$$

*(PyTorch optimizers minimize losses, so we negate to maximize)*

## 6. Critic Loss

**Run the current critic:**

```
states
  ↓
V(states)
```

**Compare against stored returns:**

$$L_{\text{value}} = \text{MSE}(V(\text{states}), \text{returns})$$

### Summary

| Network | Process |
|---|---|
| **Actor** | advantage → PPO clipped loss → update actor |
| **Critic** | returns → value loss → update critic |

## 7. One Complete PPO Update

```
              ROLLOUT
                 │
       ┌─────────┴─────────┐
       ↓                   ↓
   old_log_probs         values
       │                   │
       │                GAE/returns
       │                   │
       └─────────┬─────────┘
                 ↓
            minibatches
                 │
       ┌─────────┴─────────┐
       ↓                   ↓
 current actor         current critic
       ↓                   ↓
 new_log_probs          V(states)
       ↓                   ↓
    ratio              value loss
       ↓
 advantage
       ↓
 PPO clipping
       ↓
 actor loss
       ↓
 gradient descent
       ↓
 update both networks
       ↓
 After several epochs:
       ↓
 discard rollout
      ↓
 collect fresh experience
      ↓
    repeat
```

# Before Backpropagation: The Rollout Processing Phase

## From Rollout to Before Batching

We'll use one rollout of **8 transitions** just to keep it concrete.

---

### 1. Start with the Current Policy

We have **two networks:**

$$\text{Actor} \quad \pi_\theta(a|s)$$
$$\text{Critic} \quad V_\phi(s)$$

- **Actor:** Decides actions
- **Critic:** Estimates the value of each state

At the beginning of this rollout, both use the **current weights**.

---

### 2. Interact with the Environment

At timestep $t$:

```
state S_t
   ↓
Actor
   ↓
action A_t
   ↓
Environment
   ↓
reward R_t
   ↓
next state S_{t+1}
```

**At the same time**, the critic evaluates the state:

$$V_\phi(S_t)$$

And we calculate the probability of the **action that was actually selected**:

$$\log \pi_\theta(A_t | S_t)$$

This is saved as the **old log-probability**.

---

### 3. Store the Transition

One transition contains roughly:

```python
(
    state,
    action,
    reward,
    done,
    old_log_prob,
    value
)
```

**Example:**

```python
(
    S3,
    RIGHT,
    +2,
    False,
    -0.51,
    4.2
)
```

We repeat this for **every timestep**.

**Our rollout looks like:**

```
t=0: S0, A0, R0, done0, old_logprob0, V0
t=1: S1, A1, R1, done1, old_logprob1, V1
t=2: S2, A2, R2, done2, old_logprob2, V2
...
t=7: S7, A7, R7, done7, old_logprob7, V7
```

**At this point, everything is still in temporal order.**

---

### 4. Bootstrap Value: Next State

For TD/GAE, each transition needs:

$$V(S_t), V(S_{t+1})$$

We already stored $V(S_t)$ during rollout.

For the **final state after the rollout**, we may need a bootstrap value: $V(S_8)$

**Two cases:**

| Scenario | Bootstrap Value |
|---|---|
| Episode terminated | $V(S_8) = 0$ |
| Episode still continuing | $V(S_8) = \text{critic}(S_8)$ |

**This distinction is important.**

---

### 5. Calculate TD Errors

Now we use the **ordered trajectory**.

For every timestep:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

**Example:**

```
S3 → S4
reward = +2
V(S3) = 4.2
V(S4) = 5.0
γ = 0.9

δ3 = 2 + 0.9(5.0) - 4.2 = 2.3
```

**Interpretation:** The critic was **underestimating** what happened from $S_3$.

---

### 6. Calculate GAE Advantages

Now we take those ordered TD errors and **work backward**.

$$A_t = \delta_t + \gamma \lambda A_{t+1}$$

**So if our TD errors were:**

```
δ0, δ1, δ2, δ3, ..., δ7
```

**We calculate backward:**

```
A7 ← δ7
A6 ← δ6 + γλA7
A5 ← δ5 + γλA6
...
A0 ← δ0 + γλA1
```

**Why order matters:** We **cannot shuffle the rollout before this point**. GAE **needs the temporal sequence**.

---

### 7. Calculate Returns / Value Targets

The critic also needs a **target to learn toward**.

**Conceptually:**

$$\text{return} = \text{advantage} + V(\text{state})$$

So:

$$R_t = A_t + V(s_t)$$

**These returns become the targets for the critic** during PPO optimization.

**Now we have:**

```
state
action
old_log_prob
advantage
return
```

for every transition.

---

### 8. What Exactly Do We Have Now?

Suppose our rollout has **2000 transitions**.

**Before batching, we have:**

```
states       = [S0, S1, S2, ..., S1999]
actions      = [A0, A1, A2, ..., A1999]
old_logprob  = [L0, L1, L2, ..., L1999]
advantages   = [A0, A1, A2, ..., A1999]
returns      = [R0, R1, R2, ..., R1999]
```

**Important:** Same index → same transition

```
Transition 0: (S0, A0, old_logprob0, advantage0, return0)
Transition 1: (S1, A1, old_logprob1, advantage1, return1)
Transition 2: (S2, A2, old_logprob2, advantage2, return2)
...
```

---

### 9. What Has NOT Happened Yet?

**No PPO minibatch optimization has happened yet.**

We have **NOT**:

- Calculated the new policy probabilities
- Calculated the PPO ratio
- Clipped the ratio
- Calculated actor loss
- Calculated critic loss
- Performed backpropagation
- Updated the neural-network weights

**Those happen after batching.**

---

## The Complete Pre-Batching Pipeline

This is the **critical part to remember**:

```
CURRENT ACTOR + CRITIC
          │
          ▼
   interact with env
          │
          ▼
    collect transitions
          │
          │
          ├── state
          ├── action
          ├── reward
          ├── done
          ├── old log-prob
          └── V(state)
          │
          ▼
   rollout stays ordered
          │
          ▼
   calculate V(next state)
          │
          ▼
      TD errors
          │
          ▼
         GAE
          │
          ▼
      advantages
          │
          ▼
       returns
          │
          ▼
 ┌─────────────────────────┐
 │ NOW we have everything  │
 │ needed for PPO training │
 └─────────────────────────┘
          │
          ▼
       BATCHING
```

---

## The Critical Boundary

**Before batching** = Anything requiring temporal information

**After batching** = Optimization using shuffled minibatches

# PPO Optimization: The Batching & Training Phase

## Setup

We now start exactly where we stopped:

```
Rollout
→ TD errors
→ GAE advantages
→ returns
→ 2000 ordered transitions
→ shuffle / minibatches
```

**Assumptions:**

```
rollout = 2000 transitions
minibatch = 256
PPO epochs = 4
```

---

## 1. Create Minibatches

We now have **2000 complete transition records:**

```python
(state, action, old_log_prob, advantage, return)
```

We **shuffle their indices and divide them:**

```
2000 transitions
      ↓
256 + 256 + 256 + ... + remaining
```

**Important:** The transitions themselves are **NOT being recomputed**. Their:

```
advantage
return
old log-prob
```

are **already fixed**.

---

## 2. Take Minibatch #1

Suppose it contains **256 transitions**.

For every state in this minibatch, we **run the current actor**.

**Important:** The actor has **potentially changed** since the rollout was collected.

It produces a **probability distribution:**

```
state → current policy → probabilities
```

**Example:**

| Time | LEFT | RIGHT |
|---|---|---|
| Old (rollout) | 0.30 | 0.70 |
| Current (now) | 0.40 | 0.60 |

---

## 3. Get the New Log-Probability

We **don't care about all action probabilities** for the PPO loss.

We need the probability of the **action that was actually taken**.

If the stored action was `RIGHT`:

```python
new_log_prob = log(0.60)
```

We already have (from rollout):

```python
old_log_prob = log(0.70)
```

---

## 4. Calculate the PPO Ratio

Now:

$$\text{ratio} = \frac{\pi_{\text{new}}(a|s)}{\pi_{\text{old}}(a|s)}$$

Since we're storing **log-probabilities:**

$$\text{ratio} = e^{\text{new\_log\_prob} - \text{old\_log\_prob}}$$

**Example:**

```
old probability = 0.70
new probability = 0.60

ratio = 0.60 / 0.70 ≈ 0.857
```

**Interpretation:** The current policy **reduced the probability of this action by about 14.3%**.

---

## 5. Bring in the Stored Advantage

Remember: `advantage` was **already calculated before batching** using GAE.

Suppose for this transition:

```
advantage = +2
```

**Meaning:** This action was **better than the critic expected**.

So we'd ideally want its **probability to increase**, not decrease.

Our ratio is currently `0.857`, which means we're **moving in the wrong direction**.

**PPO's objective captures that.**

---

## 6. Calculate the Unclipped Objective

For each transition:

$$L_{\text{unclipped}} = \text{ratio} \times \text{advantage}$$

**Example:**

```
ratio     = 0.857
advantage = +2

unclipped = 1.714
```

---

## 7. Calculate the Clipped Objective

With: $\epsilon = 0.2$

We **restrict the ratio** to:

$$0.8 \leq \text{ratio} \leq 1.2$$

So:

```python
clipped_ratio = clip(0.857, 0.8, 1.2) = 0.857
```

Therefore:

$$\text{clipped objective} = 0.857 \times 2 = 1.714$$

**For this example:** Clipping does **nothing** because the ratio is already inside the range.

---

## 8. Why Clipping Matters

Take **another transition:**

```
old probability = 0.50
new probability = 0.80
ratio = 1.6
advantage = +2
```

**Unclipped:**

$$1.6 \times 2 = 3.2$$

**Clipped:**

$$\text{clip}(1.6, 0.8, 1.2) = 1.2$$
$$1.2 \times 2 = 2.4$$

PPO takes:

$$\min(3.2, 2.4) = 2.4$$

**The objective says:** "I will reward this improvement, but I won't give you extra incentive for pushing the policy this far."

---

## 9. Calculate for All 256 Samples

Now we have, for **every sample:**

```
ratio
advantage
unclipped objective
clipped objective
```

**The PPO objective takes the minimum element-wise:**

```
sample 1 → min(...)
sample 2 → min(...)
...
sample 256 → min(...)
```

Then **take the mean** across the minibatch:

```
256 objective values
        ↓
      mean
        ↓
PPO surrogate objective
```

---

## 10. Convert Objective into Actor Loss

PyTorch optimizers **minimize losses**.

But PPO's objective is something we want to **maximize**.

So:

$$\text{actor\_loss} = -\text{mean}(\min(\text{unclipped}, \text{clipped}))$$

Then:

```
actor_loss
    ↓
backpropagation
    ↓
actor gradients
    ↓
optimizer.step()
    ↓
ACTOR WEIGHTS UPDATED
```

**That is one actor update for one minibatch.**

---

## 11. Now Update the Critic

The critic gets the **same minibatch's states**.

We run:

```
state → current critic → V_new(state)
```

**Suppose:**

```
current critic: V(S) = 4.5
stored return: return = 7.0
```

**Critic loss measures the difference:**

$$L_{\text{value}} = (V(s) - \text{return})^2$$

So:

```
(4.5 - 7.0)² = 6.25
```

**Across 256 samples:**

```
256 value losses
       ↓
     mean
       ↓
critic loss
       ↓
backprop
       ↓
critic optimizer.step()
       ↓
CRITIC WEIGHTS UPDATED
```

---

## 12. Actor and Critic Updated from Minibatch

One minibatch produces:

```
                 256 transitions
                       │
          ┌────────────┴────────────┐
          ↓                         ↓
       CURRENT                    CURRENT
        ACTOR                      CRITIC
          ↓                         ↓
   new log-probs                 V(states)
          ↓                         ↓
       ratio                    compare
          ↓                    with returns
     advantages                    ↓
          ↓                    value loss
   PPO clipping                    │
          ↓                         │
      actor loss                    │
          ↓                         ↓
       backward                  backward
          ↓                         ↓
   actor optimizer            critic optimizer
          ↓                         ↓
    update actor              update critic
```

---

## 13. Then Minibatch #2

We take the **next 256 transitions**.

**Crucially:** The actor and critic now contain the **updated weights from minibatch #1**.

So we repeat:

```
Batch 2
 ↓
current actor (UPDATED)
 ↓
new log-probs
 ↓
ratio against SAME frozen old log-probs
 ↓
PPO loss
 ↓
update
```

**Critic similarly:**

```
Batch 2
 ↓
current critic (UPDATED)
 ↓
value predictions
 ↓
compare with SAME stored returns
 ↓
value loss
 ↓
update
```

Then Batch 3, Batch 4, etc.

---

## 14. Finish the First PPO Epoch

Eventually **all 2000 transitions have been used once:**

```
Epoch 1:

Batch 1 → update
Batch 2 → update
Batch 3 → update
...
Batch 8 → update
```

**Now one epoch is complete.**

But PPO **doesn't necessarily throw the rollout away yet**.

---

## 15. Repeat the Same Data for Another Epoch

We can **reuse the same 2000 transitions**.

**For Epoch 2:**

```
same states
same actions
same rewards
same advantages
same returns
same old log-probs
```

But:

```
CURRENT ACTOR → different weights
CURRENT CRITIC → different weights
```

**Therefore** the current actor produces **different probabilities**.

**Example:**

```
Epoch 1:
old probability = 0.50
current = 0.60
ratio = 1.20

After updates:

Epoch 2:
old probability = 0.50
current = 0.70
ratio = 1.40
```

**The denominator still stays 0.50.** That's the whole point of the **frozen old policy**.

---

## 16. Four Epochs

Our example:

```
2000 transitions
256 minibatch
4 epochs
```

means roughly:

```
Epoch 1:
  Batch 1 → update
  Batch 2 → update
  ...
  Batch 8 → update

Epoch 2:
  Batch 1 → update
  ...
  Batch 8 → update

Epoch 3:
  Batch 1 → update
  ...
  Batch 8 → update

Epoch 4:
  Batch 1 → update
  ...
  Batch 8 → update
```

**So the same rollout can produce many gradient updates.**

But the `old log-probabilities`, `advantages`, and `returns` **remain fixed throughout these epochs**.

---

## 17. Why Don't We Keep Using the Rollout Forever?

Because the data was **generated by the old policy**.

After many updates, the current policy can become **substantially different**.

Eventually the collected experience is **no longer a good representation** of what the current policy would experience.

So after the configured PPO epochs:

```
OLD ROLLOUT
     ↓
DISCARD
```

---

## 18. Collect a Completely New Rollout

The **updated actor** now interacts with the environment:

```
new actor
   ↓
environment
   ↓
new states/actions/rewards
   ↓
new old_log_probs
   ↓
new values
   ↓
new GAE
   ↓
new returns
```

**Then the entire process repeats.**

---

## Full PPO Pipeline

**This is the complete thing:**

```
                    INITIAL POLICY
                         │
                         ▼
                ┌─────────────────┐
                │   ENVIRONMENT   │
                └────────┬────────┘
                         │
                         ▼
                    ROLLOUT
                         │
              store transitions
                         │
        ┌────────────────┼────────────────┐
        │                │                │
      states          actions          rewards
        │                │                │
        └────────────────┼────────────────┘
                         │
                         ▼
                 critic V(s), V(s')
                         │
                         ▼
                    TD errors
                         │
                         ▼
                       GAE
                         │
                         ▼
                    advantages
                         │
                         ▼
                     returns
                         │
                         ▼
               ─── BATCHING ───
                         │
                ┌────────┴────────┐
                ▼                 ▼
             ACTOR             CRITIC
                │                 │
         current log-prob      V_new(s)
                │                 │
                ▼                 ▼
          ratio = new/old     compare
                │             with return
                ▼                 │
        PPO clipping              │
                │                 │
                ▼                 ▼
          actor loss          value loss
                │                 │
                ▼                 ▼
          backprop             backprop
                │                 │
                ▼                 ▼
          update actor        update critic
                │                 │
                └────────┬────────┘
                         ▼
                  next minibatch
                         │
                         ▼
                   next epoch
                         │
                         ▼
                configured epochs
                         │
                         ▼
                 DISCARD ROLLOUT
                         │
                         ▼
                 COLLECT NEW DATA
                         │
                         └──────────→ repeat
```

---

## The Three Things You MUST Keep Separate

### 1. Rollout Calculations

**Need temporal ordering:**

```
rewards + values
→ TD errors
→ GAE
→ advantages/returns
```

### 2. PPO Optimization

**Can use shuffled minibatches:**

```
current policy
+ frozen old log-probs
+ frozen advantages/returns
→ losses
→ gradient updates
```

### 3. New Rollout

**After several epochs:**

```
throw the old data away
→ get fresh experience
```

---

## Summary

That is the **complete PPO flow:**

$$\text{environment} \to \text{rollout} \to \text{GAE} \to \text{minibatching} \to \text{actor/critic updates} \to \text{fresh rollout}$$

# Final Mental Model: Putting It All Together

## Student Clarification: Two Important Corrections

Yes — you're **very close**, but there are **two important corrections**:

1. **Ratio is NOT calculated during the rollout.** We only store `old_log_prob`.
2. **Advantage is calculated during the rollout-processing phase, before batching.**

Let's pin the exact flow.

---

## Phase 1: Collect Rollout

Actor + critic interact with the environment for **2000 transitions**.

For **each transition** we store:

```text
state
action
reward
done
old_log_prob   ← actor's probability at collection time
value          ← critic's V(state) at collection time
```

**No weight updates yet.**

---

## Phase 2: Process the Rollout

Now the **2000 transitions are still sequential**.

We calculate:

```text
TD errors
    ↓
GAE
    ↓
advantages
    ↓
returns
```

So now **each transition has:**

```text
state
action
old_log_prob
advantage
return
```

### ⚠️ Important

We **do NOT calculate the PPO ratio yet**.

**Why?**

Because ratio needs:

```text
new_log_prob
```

and `new_log_prob` comes from running the **current actor during training**.

---

## Phase 3: Split into Minibatches

Now:

```text
2000 transitions
       ↓
shuffle
       ↓
256
256
256
...
```

**Each minibatch already contains:**

```text
state
action
old_log_prob
advantage
return
```

**The advantage doesn't need to be recalculated.**

---

## Phase 4: Process Batch 1

Now we **run the current actor and critic** on those **256 states**.

### Actor

**One batched forward pass:**

```text
256 states
    ↓
Actor
    ↓
256 probability distributions
```

From those distributions, we **extract the probability/log-probability** of the **256 actions that were actually taken**.

So:

```text
new_log_probs = 256 values
```

Then:

```text
ratio = exp(new_log_prob - old_log_prob)
```

for all 256 samples.

### Critic

At the same time, **one batched forward pass** through the critic:

```text
256 states
    ↓
Critic
    ↓
256 V(s) predictions
```

Then **compare those 256 predictions** against the **stored returns**.

---

## Phase 5: Calculate Losses

Now we have:

```text
old_log_probs  ← stored
new_log_probs  ← current actor
advantages     ← stored
returns        ← stored
new_values     ← current critic
```

### Actor Loss

**Calculate:**

```text
ratio
 ↓
ratio × advantage
 ↓
clipping
 ↓
PPO objective
```

### Critic Loss

**Calculate:**

```text
new_values vs returns
 ↓
value loss
```

---

## Where Does the Mean Happen?

**Here.**

Suppose the 256 samples produce **256 individual actor losses:**

```text
L1
L2
L3
...
L256
```

We **don't perform 256 separate gradient updates**.

We **calculate:**

```text
mean(L1 ... L256)
```

That gives **one scalar minibatch loss**.

**Same idea for the critic:**

```text
256 individual value errors
        ↓
      mean
        ↓
critic loss
```

Then:

```text
actor loss + critic loss
        ↓
backpropagation
        ↓
optimizer.step()
```

**That's one gradient update for Batch 1.**

---

## Then Batch 2

Now **the weights have changed**.

So:

```text
Batch 2
   ↓
current actor forward pass (UPDATED)
   ↓
new log_probs
   ↓
ratio against SAME old_log_probs
   ↓
PPO loss
   ↓
mean over 256 samples
   ↓
backprop
   ↓
update
```

**Critic similarly** gets its new predictions and updates against the **same stored returns**.

---

## The Exact Mental Model

```text
ROLLOUT
2000 transitions
     ↓
NO updates


PROCESS
TD errors → GAE → advantages
             ↓
          returns
     ↓
NO updates


BATCHING
2000 → 256-sized minibatches


BATCH 1
256 states
   ↓
one batched actor forward pass
one batched critic forward pass
   ↓
new log_probs + new values
   ↓
ratio + PPO loss + value loss
   ↓
MEAN across 256 samples
   ↓
backprop
   ↓
UPDATE


BATCH 2
   ↓
new forward passes using UPDATED weights
   ↓
mean loss
   ↓
update

...
```

**And the old log-probs, advantages, and returns remain frozen throughout all batches and all PPO epochs.**

**That's the clean separation.**